In [1]:
import requests
import pandas as pd
import os

# Create folder for raw data
os.makedirs("raw_data", exist_ok=True)

# Scheme names and codes
schemes = {
    "HDFC_Top100_Direct": 125497,
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_LargeCap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for fund_name, scheme_code in schemes.items():

    print(f"\n{'='*60}")
    print(f"Processing: {fund_name}")
    print(f"{'='*60}")

    try:
        # Fetch data
        url = f"https://api.mfapi.in/mf/{scheme_code}"
        response = requests.get(url)

        if response.status_code != 200:
            print("Failed to fetch data")
            continue

        data = response.json()

        # Convert NAV history to DataFrame
        df = pd.DataFrame(data['data'])

        # Save raw CSV
        csv_file = f"raw_data/{fund_name}.csv"
        df.to_csv(csv_file, index=False)

        # Dataset information
        print(f"\nSaved: {csv_file}")
        print("\nShape:")
        print(df.shape)

        print("\nData Types:")
        print(df.dtypes)

        print("\nFirst 5 Rows:")
        print(df.head())

        print("\nMissing Values:")
        print(df.isnull().sum())

        print("\nDuplicate Rows:")
        print(df.duplicated().sum())

        # Basic anomaly checks
        print("\nPossible Anomalies:")

        if df.isnull().sum().sum() > 0:
            print("- Missing values found")
        else:
            print("- No missing values")

        if 'date' in df.columns:
            print("- Date column is stored as text (object), may need datetime conversion")

        if 'nav' in df.columns:
            print("- NAV column may require numeric conversion")

    except Exception as e:
        print(f"Error processing {fund_name}: {e}")

print("\nAll datasets processed successfully.")


Processing: HDFC_Top100_Direct

Saved: raw_data/HDFC_Top100_Direct.csv

Shape:
(3102, 2)

Data Types:
date    object
nav     object
dtype: object

First 5 Rows:
         date        nav
0  16-06-2026  198.61520
1  15-06-2026  198.03200
2  12-06-2026  196.55770
3  11-06-2026  192.67220
4  10-06-2026  193.43710

Missing Values:
date    0
nav     0
dtype: int64

Duplicate Rows:
0

Possible Anomalies:
- No missing values
- Date column is stored as text (object), may need datetime conversion
- NAV column may require numeric conversion

Processing: SBI_Bluechip

Saved: raw_data/SBI_Bluechip.csv

Shape:
(3247, 2)

Data Types:
date    object
nav     object
dtype: object

First 5 Rows:
         date        nav
0  16-06-2026  105.84550
1  15-06-2026  105.81250
2  12-06-2026  105.68640
3  11-06-2026  105.57370
4  10-06-2026  105.65690

Missing Values:
date    0
nav     0
dtype: int64

Duplicate Rows:
0

Possible Anomalies:
- No missing values
- Date column is stored as text (object), may need da

In [4]:
#explore fund_master
#validate nav_history
import pandas as pd
import requests
import os



fund_master = pd.read_csv("01_fund_master.csv")
nav_history = pd.read_csv("02_nav_history.csv")

print("="*60)
print("FUND MASTER")
print("="*60)

print("Shape:", fund_master.shape)
print("\nDtypes:")
print(fund_master.dtypes)

print("\nHead:")
print(fund_master.head())

print("\nMissing Values:")
print(fund_master.isnull().sum())

print("\nDuplicates:")
print(fund_master.duplicated().sum())

# ==========================
# EXPLORE FUND MASTER
# ==========================

print("\nUnique Fund Houses:")
print(fund_master['fund_house'].unique())

print("\nUnique Categories:")
print(fund_master['category'].unique())

print("\nUnique Subcategories:")
print(fund_master['subcategory'].unique())

print("\nRisk Grades:")
print(fund_master['risk_grade'].unique())

# ==========================
# VALIDATE AMFI CODES
# ==========================

master_codes = set(fund_master['scheme_code'])
nav_codes = set(nav_history['scheme_code'])

missing_codes = master_codes - nav_codes

print("\nAMFI CODE VALIDATION")
print("Missing Codes:", len(missing_codes))

if len(missing_codes) == 0:
    print("All scheme codes validated successfully.")
else:
    print("Missing codes found:")
    print(list(missing_codes)[:20])

# ==========================
# LIVE NAV FETCH
# ==========================

os.makedirs("raw_data", exist_ok=True)

schemes = {
    "HDFC_Top100_Direct": 125497,
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_LargeCap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for name, code in schemes.items():

    url = f"https://api.mfapi.in/mf/{code}"

    response = requests.get(url)

    if response.status_code == 200:

        data = response.json()

        df = pd.DataFrame(data["data"])

        df.to_csv(f"raw_data/{name}.csv", index=False)

        print(f"\n{name} saved.")
        print("Shape:", df.shape)

    else:
        print(f"{name} fetch failed.")

# ==========================
# DATA QUALITY SUMMARY
# ==========================

print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

print("Fund Master Rows:", fund_master.shape[0])
print("Fund Master Columns:", fund_master.shape[1])

print("Missing Values:",
      fund_master.isnull().sum().sum())

print("Duplicate Records:",
      fund_master.duplicated().sum())

print("Missing Scheme Codes:",
      len(missing_codes))

print("\nDay 1 Completed Successfully")

FUND MASTER
Shape: (40, 15)

Dtypes:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

Head:
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_ca

KeyError: 'subcategory'